In [1]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from youtube_transcript_api import YouTubeTranscriptApi
from langchain_text_splitters import RecursiveCharacterTextSplitter
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings

/Users/ashutosh/projects/LangChain/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/var/folders/vn/j3kyb3f14dngz6y0dp2yf5jh0000gn/T/ipykernel_92560/3053815667.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [2]:
load_dotenv()

#1.a. DataLoader
#Load the transcript as Document via youtube-transcript-api 
import re
def manual_ytLoader(id_or_URL: str) -> Document:
    # Regex to handle regular URLs, shorts, youtu.be, and raw IDs
    pattern = r'(?:v=|\/shorts\/|\/embed\/|\/vi\/|youtu\.be\/|^\b)([a-zA-Z0-9_-]{11})'
    match = re.search(pattern, id_or_URL)
    
    if match:
        video_id = match.group(1)
    else:
        video_id = id_or_URL  # Fallback to the input string
    try:
        # 1. Initialize the API instance
        api_client = YouTubeTranscriptApi()
        
        # 2. Use .fetch() to get the transcript data
        raw_transcript = api_client.fetch(video_id)

        clean_text = " ".join(doc.text for doc in raw_transcript)

        if clean_text.strip():
            return Document(
                page_content=clean_text, 
                metadata={"source": f"<https://youtube.com/watch?v={video_id}>", "video_id": video_id}
            )
    except Exception as e:
        print(f"File Transcript cannot be loaded: {e} \n")

docs = manual_ytLoader("https://www.youtube.com/watch?v=4ZQkYSpmOdU")

In [3]:
#1.b TextSplitter
splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 200
)

chunked_docs = splitter.split_documents([docs])

In [ ]:
#1.c initializing the vector_store

#first embedding model
embd = HuggingFaceEmbeddings(model="sentence-transformers/all-MiniLM-L6-v2")

vector_Store = FAISS.from_documents(
    documents= chunked_docs,
    embedding=embd
)

#2 Retrieval
retriever = vector_Store.as_retriever(search_kwargs = {"k":3})


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6617.75it/s]


In [5]:
#3. PromptTemplate Building

prompt = ChatPromptTemplate.from_template(
    """
    you are a helpful assistant. Help me answer the query : {query} , strictly with ONLY reference to
    the given Youtube transcript context : {context},
    If the context doesn't provide sufficient or relevant information simple say 'I don't know' or mention 
    that the video does not talk about the given query.
"""
)


In [6]:
# Defining a func to Merge all the recieved context docs into one single string
#to later make it a runnable and include in our LCEL

def contextMerger(docs : list[Document]) -> str:
    return " " \
    "\n\n".join(doc.page_content for doc in docs)

In [12]:
#4. Generation

query = "Does this video talk about stress management?"

context = contextMerger(retriever.invoke(query))

final_prompt = prompt.invoke({"query": query, "context" : context })

In [13]:
#create a string output parser
from langchain_core.output_parsers import StrOutputParser
parser = StrOutputParser()

In [14]:
print(final_prompt)

messages=[HumanMessage(content="\n    you are a helpful assistant. Help me answer the query : Does this video talk about stress management? , strictly with ONLY reference to\n    the given Youtube transcript context : machine, you have not bothered to read even the user's manual, you want to just blunder around. No? Young \npeople, it's time you figure out a few things about you, if you don't know how, we will give you tools how to \nfigure this machine out because in your life many issues will come, more issues come up in your life means\nyou're living a more active life. Nothing came up means you're not living. Yes, lots of issues every day. I have the \nmaximum number of trouble going on in my life on a daily basis because so much of activity around the \nworld, global level of activity only with volunteers, okay? Volunteers means nobody is qualified for the job and \nyou can't fire them for inefficiency and they love you, what to do? So, this one thing you must fix that is \nin you

In [15]:
model = ChatGroq(model = "llama-3.3-70b-versatile")

res = parser.invoke(model.invoke(final_prompt))

In [16]:
print(res)

The video does talk about stress management indirectly. The speaker discusses how people face many issues in life and that it's a natural part of living an active life. They also mention that one should learn to "fix" their problems and not let them become the issue. Additionally, the speaker touches on the idea that people often suffer not from their actual life, but from their memories and imagination, implying that managing one's thoughts and emotions is important. However, the video does not provide a direct or comprehensive discussion on stress management techniques.


In [33]:
# Building Chain

from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda

parall = RunnableParallel({
    "query" : RunnablePassthrough(),
    "context" : retriever | RunnableLambda(contextMerger)
})

forward_chain = prompt | model | parser

final_chain = parall | forward_chain

In [35]:
res = final_chain.invoke("Summarize the video in short")
print(res)

The video appears to be a conversation about the possibility of creating a deeper, simpler explanation of physics that goes beyond the standard model. The speaker mentions that this new explanation could provide glimpses into mysteries such as consciousness, life, and gravity. The conversation also touches on the idea of creating an artificial general intelligence (AGI) system that achieves human-level intelligence and beyond. If given the chance to ask the AGI system one question, the speaker would ask "what is the true nature of reality?" However, the video does not provide a clear answer to this question, instead, it shows the AGI system struggling to explain it.
